# NIPOPETO — chẩn đoán kiến trúc trên Kaggle CPU

Notebook này dùng protocol `architecture-diagnostics-v2`, độc lập baseline B1 16 lượt đã đạt. Mục tiêu là kiểm chứng rủi ro thấy trong code: dashboard dùng tổng tại DB và preview có giới hạn; danh sách đọc trang 1 với 20 dòng; admin và chủ nuôi dùng chung API/DB. Kết quả có thể đạt hoặc vượt ngưỡng, không mặc định monolith có lỗi.

Commit/push code harness, điền SHA mới, chọn Accelerator **None**, Internet **On**. Mỗi run tạo DB tạm, không dùng dữ liệu thật. Chọn một `SUITE`, chạy các cell theo thứ tự. Sau khi dừng vì vượt ngưỡng/lỗi, vẫn chạy cell export cuối. Giữ ZIP cũ.

Thứ tự đề xuất: `data` (~40–50 phút), `interference` (~25–35 phút), `stress` (~60–75 phút), `soak` (~35 phút). Thời gian còn phụ thuộc setup/build/seed. Không cần chạy mọi suite ngay; xem kết quả từng suite trước.

Bản phân trang dùng collection `items/pagination`, trang 1 với 20 dòng và bootstrap `view=dashboard`. Chưa có phép đo Kaggle cho bản này. ZIP protocol cũ là lịch sử, không so trực tiếp bulk read với page read.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, pwd, re, shutil, signal, subprocess, sys, urllib.request, uuid, zipfile

REPO_URL = "https://github.com/Nguyenquanghong/Pet-healthcare.git"
COMMIT_SHA = "DIEN_FULL_SHA_40_KY_TU_SAU_KHI_PUSH"
RUN_LABEL = "B1D"  # B2D cho bản cải tiến, cùng harness/cấu hình và phiên CPU.
SUITE = "data"  # data | interference | stress | soak
P95_LIMIT_MS = 2000  # Ngưỡng thăm dò từng API, chưa phải SLA nghiệp vụ.
OWNER_PROBE_LIMIT_MS = 500  # Nhạy hơn cho GET pets của nhóm chủ nuôi đối chứng.
PROTOCOL = "architecture-diagnostics-v2"
WORKING = Path("/kaggle/working")
assert WORKING.is_dir(), "Notebook này phải chạy trên Kaggle."
assert not os.environ.get("COLAB_GPU") and not Path("/dev/nvidia0").exists(), "Chọn Accelerator: None."
assert re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA), "Điền full SHA của bản đã push."
assert re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\.git", REPO_URL), "Dùng GitHub public, không nhúng token."
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_LABEL)
assert SUITE in ("data", "interference", "stress", "soak")
assert 0 < P95_LIMIT_MS < float("inf") and 0 < OWNER_PROBE_LIMIT_MS < float("inf")
repo = WORKING / "Pet-healthcare"
print("Suite:", SUITE, "| protocol:", PROTOCOL, "| CPU logic:", os.cpu_count())


## Lấy đúng source

Dùng lại checkout sạch nếu đã clone; có thể chuyển sang SHA mới khi checkout sạch. Dừng khi có thay đổi để tránh mất mã hoặc gắn sai nhãn kết quả. Không tự xóa checkout cũ. Trạng thái detached HEAD khi chọn đúng SHA là bình thường.


In [ ]:
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert not dirty, "Checkout có thay đổi; hãy kiểm tra trước khi tiếp tục."
if actual.lower() != COMMIT_SHA.lower():
    subprocess.run(["git", "fetch", "origin", COMMIT_SHA], cwd=repo, check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=repo, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert actual.lower() == COMMIT_SHA.lower() and not dirty, "SHA không khớp hoặc working tree không sạch."
source_protocol = subprocess.check_output([sys.executable, "-c", "import sys; sys.path.insert(0, sys.argv[1]); from diagnostics import DIAGNOSTIC_PROTOCOL; print(DIAGNOSTIC_PROTOCOL)", str(repo / "benchmarks/kaggle")], text=True).strip()
assert source_protocol == PROTOCOL, "Source chưa chứa harness hiện tại. Commit/push toàn bộ benchmarks/kaggle (gồm các file diagnostic) rồi chọn lại SHA."
print("Source commit:", actual, "| dirty_worktree:", bool(dirty), "| protocol:", source_protocol)


## Cài công cụ và kiểm tra trước chạy

Cần Internet để tải PostgreSQL server, Node.js 24 và Python packages. Cài Python bằng chính interpreter của kernel. Cell kiểm đủ `initdb`, `pg_ctl`, `psql`, `createdb`, và tài khoản hệ thống postgres khi chạy bằng root; client PostgreSQL đơn lẻ chưa đủ.


In [ ]:
def find_pg_bin():
    candidates = []
    if shutil.which("pg_config"):
        candidates.append(Path(subprocess.check_output(["pg_config", "--bindir"], text=True).strip()))
    candidates.extend(sorted(Path("/usr/lib/postgresql").glob("*/bin"), reverse=True))
    return next((p for p in candidates if all((p / tool).is_file() for tool in ("initdb", "pg_ctl", "postgres", "psql", "createdb"))), None)

pg_bin = find_pg_bin()
if pg_bin is None or not shutil.which("pg_config"):
    assert os.geteuid() == 0, "Không có quyền cài PostgreSQL server trong phiên này."
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-client", "libpq-dev"], check=True)
    pg_bin = find_pg_bin()
assert pg_bin is not None, "Chưa tìm được PostgreSQL server; xem output cài đặt."
os.environ["BENCH_PG_BIN"] = str(pg_bin)
os.environ["PATH"] = str(pg_bin) + os.pathsep + os.environ["PATH"]
if os.geteuid() == 0:
    assert shutil.which("runuser"), "Thiếu runuser để chạy PostgreSQL bằng user postgres."
    pwd.getpwnam("postgres")

assert platform.machine() in ("x86_64", "amd64"), "Bản Node pin này dành cho Kaggle Linux x64."
node_dir = WORKING / "node24"
if (node_dir / "bin/node").is_file():
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
node_ok = shutil.which("node") and subprocess.check_output(["node", "--version"], text=True).strip().startswith("v24.")
if not node_ok:
    version = "v24.15.0"
    name = f"node-{version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/{version}/"
    archive = WORKING / name
    urllib.request.urlretrieve(base + name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60).read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node checksum không khớp."
    node_dir.mkdir(exist_ok=True)
    subprocess.run(["tar", "-xJf", str(archive), "-C", str(node_dir), "--strip-components=1"], check=True)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "benchmarks/kaggle/requirements.txt")], check=True)
for program, args in {"node": ["--version"], "npm": ["--version"], "pg_config": ["--version"]}.items():
    print(program, subprocess.check_output([program, *args], text=True).strip())
print("PostgreSQL server:", subprocess.check_output([str(pg_bin / "postgres"), "--version"], text=True).strip())
print("Python:", sys.version.split()[0], "| Locust:", subprocess.check_output([sys.executable, "-m", "locust", "--version"], text=True).strip())
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(repo / "benchmarks/kaggle"), "-p", "test_*.py", "-v"], check=True)


## Chọn phép đo và chạy

- **data:** 3 admin, 500 chủ nuôi/thú cưng; 1.000 → 10.000 → 50.000 đợt lịch sử. Một đợt gồm lịch hẹn, bệnh án, booking khách sạn, nhật ký, hóa đơn, dòng hóa đơn và 2 thông báo = **8 bản ghi**, không phải tổng dataset chỉ có N hàng. Lịch sử là dữ liệu giả đã hoàn thành; không nhúng ảnh lớn để tạo tải giả tạo.
- **interference:** cùng 500 chủ nuôi/thú cưng và 10.000 đợt lịch sử; so 50 owner đọc GET pets với 50 owner + 5 admin đọc bootstrap. Giữ số owner cố định, hai cặp chạy theo thứ tự A/B rồi B/A. Xem p95/RPS/lỗi **OWNER probe**, không dùng p95 tổng để kết luận owner bị ảnh hưởng.
- **stress:** cùng dữ liệu 10.000 đợt, cùng 500 chủ nuôi/thú cưng; tăng MixedUser 100/200/300/500. Mix giống baseline (GET + tạo lịch + báo tìm thấy), giữ think time 0,5–1,5s. Mỗi mức lặp hai lần.
- **soak:** MixedUser 100 trong khoảng 30 phút sau ramp; theo dõi resources.json và load_stats_history.csv theo thời gian. Chạy sau khi đã biết mức tải ổn định; một lượt này không đủ khẳng định rò rỉ bộ nhớ.

Admin nghỉ 2–4s giữa tác vụ; owner nghỉ 0,5–1,5s. Mỗi run có DB mới, seed ngoài thời gian đo, ANALYZE sau seed. Warmup đủ ramp+15s, process đo có ít nhất 300s sau ramp (soak 1800s). **Số tổng hợp vẫn gồm ramp-up**, không phải riêng steady state. Bootstrap lớn cũng có thể làm Locust bận nhận/parse JSON; kiểm tra CPU generator, byte/response và DB trước khi kết luận API là nút thắt.

Ngưỡng sau mỗi run: lỗi >1%, p95 API >P95_LIMIT_MS hoặc owner probe >OWNER_PROBE_LIMIT_MS. Dừng suite ở lần vượt ngưỡng/lỗi đầu; không tự tăng tải tiếp. Ngưỡng này dùng thăm dò, không phải cam kết nghiệp vụ.


In [ ]:
current_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
current_dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
assert current_sha.lower() == COMMIT_SHA.lower() and not current_dirty, "Source thay đổi sau clone."
plan_code = "import json,sys; sys.path.insert(0,sys.argv[1]); from diagnostics import build_plan; print(json.dumps(build_plan(sys.argv[2])))"
PLAN = json.loads(subprocess.check_output([sys.executable, "-c", plan_code, str(repo / "benchmarks/kaggle"), SUITE], text=True))
evidence = WORKING / f"nipopeto-diagnostic-{RUN_LABEL}-{SUITE}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:8]}"
evidence.mkdir(exist_ok=False)
run_error = None
run_count = 0
try:
    for case in PLAN:
        folder = evidence / f"{RUN_LABEL}_{case['name']}"
        args = [sys.executable, str(repo / "benchmarks/kaggle/runner.py"), "--repo", str(repo), "--output", str(folder),
                "--diagnostic", "--profile", case["profile"], "--users", str(case["users"]), "--spawn-rate", str(case["spawn_rate"]),
                "--warmup", f"{case['warmup_seconds']}s", "--duration", f"{case['duration_seconds']}s",
                "--fixture-users", str(case["fixture_users"]), "--history-rows", str(case["history_rows"]),
                "--owner-users", str(case["owner_users"]), "--admin-users", str(case["admin_users"]),
                "--p95-limit-ms", str(P95_LIMIT_MS), "--owner-probe-limit-ms", str(OWNER_PROBE_LIMIT_MS)]
        launcher_log = evidence / f"{folder.name}-runner.log"
        print("Running", case, "| log:", launcher_log.name, flush=True)
        run_count += 1
        with launcher_log.open("w", encoding="utf-8") as log:
            process = subprocess.Popen(args, cwd=repo, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            try:
                returncode = process.wait()
            except KeyboardInterrupt:
                if process.poll() is None:
                    try:
                        os.killpg(process.pid, signal.SIGINT)
                    except ProcessLookupError:
                        pass
                    process.wait(timeout=45)
                raise
        print("\n".join(launcher_log.read_text(encoding="utf-8", errors="replace").splitlines()[-8:]), flush=True)
        if returncode:
            run_error = f"{folder.name}: exit {returncode}; xem manifest, endpoint_final.json và runner log."
            break
except KeyboardInterrupt:
    run_error = "Đã dừng; kiểm tra cleanup của runner trước khi chạy tiếp."
except Exception as error:
    run_error = f"{type(error).__name__}: {error}. Nếu runner chưa dọn xong, kiểm tra log hoặc dùng phiên mới."
finally:
    (evidence / "notebook-summary.json").write_text(json.dumps({
        "commit": actual, "label": RUN_LABEL, "suite": SUITE, "benchmark_protocol": PROTOCOL, "plan": PLAN,
        "p95_limit_ms": P95_LIMIT_MS, "owner_probe_limit_ms": OWNER_PROBE_LIMIT_MS,
        "attempted_runs": run_count, "expected_runs": len(PLAN), "stop_reason": run_error,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã thử:", run_count, "/", len(PLAN), "| Bằng chứng:", evidence, "| Lý do dừng:", run_error)


## Xuất ZIP và đọc kết quả

Luôn chạy cell cuối, kể cả khi suite dừng. `COMPLETED_WITHIN_LIMITS` nghĩa là đủ ma trận đã chọn và dưới ngưỡng; `LIMIT_REACHED` nghĩa là đã có run đo vượt ngưỡng; `RUN_ERROR_OR_PARTIAL` cần đọc lỗi/setup/warmup. Không gọi việc tìm thấy giới hạn tải là đã chứng minh cần microservice.

ZIP chứa `endpoint-summary.csv` (p50/p95/p99, lỗi, RPS và byte trung bình từng response), final counters, dữ liệu tài nguyên, fixture-summary không token và logs đã lọc. `notebook-summary.json` có so sánh owner khi có/không có admin nếu đủ cặp hợp lệ. Không cộng RPS nhiều lượt, không gộp p95 bằng phép trung bình. Xem lịch sử CSV và resources để phân biệt độ trễ cao liên tục với đỉnh lúc ramp/khởi động.

Tải ZIP từ **Output**, refresh nếu chưa thấy. Với B1D/B2D phải dùng cùng protocol/plan/ngưỡng/tổng CPU/RAM/versions trong cùng phiên Kaggle. Nếu đổi API sang phân trang, thiết kế lại cùng hành trình đọc cho cả hai bản, không so toàn bộ danh sách với một trang rồi nhận là nhanh hơn.


In [ ]:
assert "evidence" in globals() and evidence.is_dir(), "Chạy cell suite trước."
subprocess.run([sys.executable, str(repo / "benchmarks/kaggle/diagnostic_report.py"), str(evidence)], check=True)
print("Tải ZIP vừa in ở trên từ mục Output. Giữ cả kết quả vượt ngưỡng để phân tích.")
